<a href="https://colab.research.google.com/github/Zikirullo/clear-vision/blob/develop/notebooks/01_download_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Download datasets

Downloads every raw dataset into `MyDrive/clear_vision/datasets/raw/<name>/`.

- **Safe to re-run.** A dataset is downloaded to the Colab disk first, copied to Drive,
  and only then marked with a `.done` file. Finished datasets are skipped. A folder
  without `.done` is a partial download and is replaced on the next run.
- **Before running:** add `ROBOFLOW_API_KEY` under Secrets (the key icon on the left)
  and turn on notebook access. For section 5, upload the Mendeley `.zip` files to
  `MyDrive/clear_vision/datasets/raw/mendeley/`.
- No GPU is needed. Run all cells, then check the summary table at the end.


In [ ]:
# Settings: edit these lists, then run all cells.

# (workspace, project, version, folder name under datasets/raw/)
# If a version fails, the project's latest version is used instead.
ROBOFLOW_DATASETS = [
    ("blindspot-fmcu0", "sidewalk-obstacles", 1, "rf_sidewalk_obstacles"),
    ("blindwalk", "bollard-v16ec", 1, "rf_bollard"),
    ("object-detection-3n5oe", "final-detection-9g3ji", 6, "rf_pole_curb_stairs"),
    ("kyonggi-university-r7unh", "protection_zone", 4, "rf_scooter_bollard"),
    ("tom-lai-8bp7n", "stairs-i2yia", 3, "rf_stairs"),
    ("georgia-institute-of-technology-bwqcc", "pothole-detection-mev9q", 6, "rf_pothole"),
    ("distracted-driving-yu8re", "pedestrian-crosswalk-c0irg", 1, "rf_crosswalk"),
    ("grape-disease-prediction", "general-cone-detection-ow4vb", 2, "rf_cone"),
    ("workspace-an3qw", "manhole-j9klz", 2, "rf_manhole"),
]

# Open Images V7: class names exactly as Open Images spells them.
OI_CLASSES = ["Stairs", "Traffic light", "Fire hydrant", "Bench", "Bicycle", "Person", "Car", "Dog"]
OI_MAX_PER_CLASS = 1500  # per class and per split (train, validation)

# COCO-2017 subset.
COCO_CLASSES = [
    "person", "bicycle", "car", "motorcycle", "bus",
    "truck", "dog", "bench", "fire hydrant", "traffic light",
]
COCO_MAX_SAMPLES = {"train": 5000, "validation": 1000}  # set validation to 0 to skip it


## 1. Setup

In [ ]:
# 1. Setup: Drive, packages, paths, API key, helpers
from google.colab import drive

drive.mount("/content/drive")

%pip install -q roboflow fiftyone

import json
import os
import shutil
import time
import zipfile
from pathlib import Path

from google.colab import userdata

DRIVE_ROOT = Path("/content/drive/MyDrive/clear_vision")
RAW = DRIVE_ROOT / "datasets" / "raw"
STAGING = Path("/content/staging")  # Colab disk: much faster than writing file by file to Drive
RAW.mkdir(parents=True, exist_ok=True)
STAGING.mkdir(parents=True, exist_ok=True)

DONE = ".done"  # written last, so a folder without it is a partial download
FAILURES: dict[str, str] = {}

try:
    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
    print("ROBOFLOW_API_KEY loaded")
except Exception as e:
    ROBOFLOW_API_KEY = None
    print(f"WARNING: ROBOFLOW_API_KEY not available ({type(e).__name__}).")
    print("         Add it under Secrets (key icon) with notebook access, then re-run.")
    print("         Roboflow datasets will be skipped.")


def is_done(dest: Path) -> bool:
    return (dest / DONE).exists()


def skip_if_done(dest: Path) -> bool:
    if is_done(dest):
        print(f"[{dest.name}] already downloaded, skipping")
        return True
    if dest.exists():
        print(f"[{dest.name}] partial download found, it will be replaced")
    return False


def fresh_staging(name: str) -> Path:
    local = STAGING / name
    shutil.rmtree(local, ignore_errors=True)
    return local


def publish(local: Path, dest: Path, info: dict) -> None:
    """Copy a finished download from the Colab disk to Drive, then mark it done."""
    t0 = time.time()
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(local, dest)
    (dest / DONE).write_text(json.dumps(info, indent=2))
    shutil.rmtree(local, ignore_errors=True)
    print(f"  copied to {dest} in {time.time() - t0:.0f}s")


def record_failure(name: str, error: Exception) -> None:
    FAILURES[name] = f"{type(error).__name__}: {error}"
    print(f"  FAILED: {FAILURES[name]}")


print(f"Raw datasets folder: {RAW}")


## 2. Roboflow datasets (YOLOv8)

In [ ]:
# 2. Roboflow datasets (YOLOv8 format)
from roboflow import Roboflow


def version_number(version) -> int:
    return int(str(version.version).split("/")[-1])


def download_roboflow(rf, workspace: str, project_id: str, version: int, local: Path) -> int:
    """Download one Roboflow version; fall back to the latest one. Returns the version used."""
    project = rf.workspace(workspace).project(project_id)
    try:
        project.version(version).download("yolov8", location=str(local), overwrite=True)
        return version
    except Exception as e:
        print(f"  version {version} failed ({type(e).__name__}: {e}), trying the latest version")
    versions = project.versions()
    if not versions:
        raise RuntimeError(f"{workspace}/{project_id} has no versions")
    latest = max(version_number(v) for v in versions)
    shutil.rmtree(local, ignore_errors=True)
    project.version(latest).download("yolov8", location=str(local), overwrite=True)
    print(f"  USED LATEST VERSION {latest} instead of {version}")
    return latest


rf = Roboflow(api_key=ROBOFLOW_API_KEY) if ROBOFLOW_API_KEY else None

for workspace, project_id, version, name in ROBOFLOW_DATASETS:
    dest = RAW / name
    if skip_if_done(dest):
        continue
    print(f"\n[{name}] {workspace}/{project_id} v{version}")
    if rf is None:
        record_failure(name, RuntimeError("no ROBOFLOW_API_KEY"))
        continue
    local = fresh_staging(name)
    t0 = time.time()
    try:
        used = download_roboflow(rf, workspace, project_id, version, local)
        print(f"  downloaded v{used} in {time.time() - t0:.0f}s")
        publish(local, dest, {
            "source": "roboflow", "workspace": workspace, "project": project_id,
            "requested_version": version, "version": used, "format": "yolov8",
        })
    except Exception as e:
        record_failure(name, e)

print("\nRoboflow section finished")


## 3. Open Images V7

Loaded one class at a time so each class gets up to `OI_MAX_PER_CLASS` images, merged without duplicates, then exported with only `OI_CLASSES` boxes. The first run downloads large annotation files (about 2 GB for train).

In [ ]:
# 3. Open Images V7 (detections) -> YOLO
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F

fo.config.dataset_zoo_dir = "/content/fiftyone"  # zoo downloads stay on the Colab disk


def detections_field(collection) -> str:
    fields = collection.get_field_schema(embedded_doc_type=fo.Detections)
    if not fields:
        raise RuntimeError(f"{collection.name} has no detections field")
    return next(iter(fields))


def export_yolo(collection, classes: list[str], export_dir: Path, split: str) -> int:
    """Export only `classes` boxes (images without any are dropped). Returns image count."""
    field = detections_field(collection)
    view = collection.filter_labels(field, F("label").is_in(classes))
    view.export(
        export_dir=str(export_dir),
        dataset_type=fo.types.YOLOv5Dataset,
        label_field=field,
        split=split,
        classes=classes,
    )
    return len(view)


name = "openimages"
dest = RAW / name
if not skip_if_done(dest):
    print(f"\n[{name}] Open Images V7, {len(OI_CLASSES)} classes, up to {OI_MAX_PER_CLASS} per class")
    local = fresh_staging(name)
    t0 = time.time()
    try:
        counts = {}
        for split, yolo_split in [("train", "train"), ("validation", "val")]:
            merged = fo.Dataset(f"oi7-{split}-merged", overwrite=True)
            for cls in OI_CLASSES:
                print(f"  {split}: loading up to {OI_MAX_PER_CLASS} images with '{cls}'")
                part = foz.load_zoo_dataset(
                    "open-images-v7",
                    split=split,
                    label_types=["detections"],
                    classes=[cls],
                    max_samples=OI_MAX_PER_CLASS,
                    shuffle=True,
                    seed=51,
                    dataset_name=f"oi7-{split}-{cls}",
                    drop_existing_dataset=True,
                )
                # One image can contain several classes: keep it once.
                merged.merge_samples(part, key_field="filepath", skip_existing=True)
            counts[yolo_split] = export_yolo(merged, OI_CLASSES, local, yolo_split)
            print(f"  {split}: exported {counts[yolo_split]} images")
        print(f"  finished in {time.time() - t0:.0f}s")
        publish(local, dest, {
            "source": "open-images-v7", "classes": OI_CLASSES,
            "max_per_class": OI_MAX_PER_CLASS, "images": counts, "format": "yolov5",
        })
    except Exception as e:
        record_failure(name, e)

print("\nOpen Images section finished")


## 4. COCO-2017 subset

In [ ]:
# 4. COCO-2017 subset -> YOLO
name = "coco_subset"
dest = RAW / name
if not skip_if_done(dest):
    print(f"\n[{name}] COCO-2017, {len(COCO_CLASSES)} classes, {COCO_MAX_SAMPLES}")
    local = fresh_staging(name)
    t0 = time.time()
    try:
        counts = {}
        for split, yolo_split in [("train", "train"), ("validation", "val")]:
            max_samples = COCO_MAX_SAMPLES.get(split, 0)
            if max_samples <= 0:
                continue
            print(f"  {split}: loading up to {max_samples} images")
            subset = foz.load_zoo_dataset(
                "coco-2017",
                split=split,
                label_types=["detections"],
                classes=COCO_CLASSES,
                max_samples=max_samples,
                shuffle=True,
                seed=51,
                dataset_name=f"coco-{split}-subset",
                drop_existing_dataset=True,
            )
            counts[yolo_split] = export_yolo(subset, COCO_CLASSES, local, yolo_split)
            print(f"  {split}: exported {counts[yolo_split]} images")
        print(f"  finished in {time.time() - t0:.0f}s")
        publish(local, dest, {
            "source": "coco-2017", "classes": COCO_CLASSES,
            "max_samples": COCO_MAX_SAMPLES, "images": counts, "format": "yolov5",
        })
    except Exception as e:
        record_failure(name, e)

print("\nCOCO section finished")


## 5. Mendeley

In [ ]:
# 5. Mendeley: unzip uploaded archives in place
mendeley = RAW / "mendeley"
zips = sorted(mendeley.glob("*.zip")) if mendeley.exists() else []

if not zips:
    print(f"No .zip files in {mendeley}")
    print("Download the Mendeley datasets in a browser, upload the .zip files there, re-run this cell.")

for zip_path in zips:
    target = zip_path.with_suffix("")  # raw/mendeley/<zip name>/
    if skip_if_done(target):
        continue
    print(f"\n[mendeley/{target.name}] unzipping {zip_path.name}")
    t0 = time.time()
    try:
        shutil.rmtree(target, ignore_errors=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(target)  # extractall refuses paths that escape `target`
        (target / DONE).write_text(json.dumps({"source": "mendeley", "zip": zip_path.name}, indent=2))
        print(f"  unzipped in {time.time() - t0:.0f}s")
    except Exception as e:
        record_failure(f"mendeley/{target.name}", e)

print("\nMendeley section finished")


## 6. Summary

In [ ]:
# 6. Summary table
import pandas as pd
import yaml
from IPython.display import display

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def count_files(root: Path) -> tuple[int, int]:
    """(images, YOLO label files). Label files are .txt files inside a `labels` folder."""
    images = labels = 0
    for dirpath, _, filenames in os.walk(root):
        in_labels = "labels" in Path(dirpath).relative_to(root).parts
        for filename in filenames:
            ext = Path(filename).suffix.lower()
            if ext in IMAGE_EXTS:
                images += 1
            elif ext == ".txt" and in_labels:
                labels += 1
    return images, labels


def class_names(root: Path) -> tuple[list[str], str]:
    """Class names from data.yaml / dataset.yaml, else from a classes file."""
    for pattern in ("data.yaml", "dataset.yaml"):
        for path in sorted(root.rglob(pattern)):
            try:
                names = (yaml.safe_load(path.read_text()) or {}).get("names")
            except yaml.YAMLError:
                continue
            if isinstance(names, dict):
                names = [names[k] for k in sorted(names)]
            if names:
                return [str(n) for n in names], path.name
    for pattern in ("classes.txt", "_classes.txt", "*.names"):
        for path in sorted(root.rglob(pattern)):
            names = [line.strip() for line in path.read_text().splitlines() if line.strip()]
            if names:
                return names, path.name
    return [], "-"


def dataset_dirs() -> list[Path]:
    dirs = []
    for path in sorted(p for p in RAW.iterdir() if p.is_dir() and not p.name.startswith(".")):
        if path.name == "mendeley":
            dirs += sorted(p for p in path.iterdir() if p.is_dir() and not p.name.startswith("."))
        else:
            dirs.append(path)
    return dirs


rows = []
for root in dataset_dirs():
    name = str(root.relative_to(RAW))
    images, labels = count_files(root)
    names, source = class_names(root)
    rows.append({
        "dataset": name,
        "status": "done" if is_done(root) else "PARTIAL",
        "images": images,
        "label files": labels,
        "classes": len(names),
        "class names": ", ".join(names) if names else "(none found)",
        "classes from": source,
    })
for name, error in FAILURES.items():
    if not any(row["dataset"] == name for row in rows):
        rows.append({"dataset": name, "status": "FAILED", "images": 0, "label files": 0,
                     "classes": 0, "class names": error, "classes from": "-"})

pd.set_option("display.max_colwidth", None)
summary = pd.DataFrame(rows)
display(summary)
print(f"\n{len(rows)} datasets, {summary['images'].sum() if rows else 0} images in {RAW}")

if FAILURES:
    print(f"\n{len(FAILURES)} FAILED (re-run all cells to retry only these):")
    for name, error in FAILURES.items():
        print(f"  - {name}: {error}")
else:
    print("No failures.")
